## Factory Sensor Data Analysis Project

### Stage Two — Data Cleaning and Validation

This stage moves from assessing data quality to carrying out cleaning operations and validating the results.

The notebook addresses the identified quality issues while preserving the original data unchanged.

### Data files used

| File | Rows | Purpose |
|---|---:|---|
| `original_data.csv` | 8,000 | Original data source |
| `raw_data.csv` | 8,020 | Raw working copy used in the project |
| `cleaned_data.csv` | — | Destination for the cleaned data |

### Workflow

The raw working copy is the starting point for cleaning and validation. Appropriate treatments are then applied to the issues identified during the quality assessment. The result of each treatment is checked before accepting the resulting data. Finally, the cleaned data is saved in a file separate from the raw files.

### Methodological note

Cleaning is performed on a separate copy of the data. The original source is not modified, and the raw working copy is not overwritten. Treatments are limited to the safe, documented procedures defined in the project. Unusual values are not deleted and missing data is not imputed without a clear justification.


## Load the Raw Data and Create a Working Copy

The raw working dataset is loaded and an independent copy is created for cleaning. This preserves the raw data unchanged during subsequent processing.

### Inspection steps

Import the libraries required to read the data and perform analytical operations. Load the raw data file from the project's data directory, then create a separate DataFrame for later cleaning. Finally, inspect the raw dataset dimensions and count fully identical rows.

### Data file used

| File | Purpose |
|---|---|
| `raw_data.csv` | Raw working copy used for cleaning |

### Libraries used

- `pandas` — read and manipulate tabular data
- `numpy` — perform numerical operations where needed
- `Path` — handle file paths where needed

### Variables

- `raw` — raw DataFrame after loading the file
- `df_clean` — independent copy dedicated to cleaning

### Code elements

- `pd.read_csv()` — read the raw data file
- `copy()` — create an independent DataFrame copy
- `shape` — display the row and column counts
- `duplicated()` — identify fully identical rows
- `sum()` — count duplicate rows
- `print()` — display the initial inspection results

### Purpose of the check

This check confirms that the data was loaded and establishes its size before cleaning begins. It also verifies the duplicate count for comparison with the earlier quality assessment.

### Methodological note

An independent copy is created so that cleaning does not change the raw DataFrame. Copying data and checking duplicates do not delete records or alter values. Importing libraries does not imply that all their functions are used in this step. No treatment is applied during loading and the initial inspection.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

raw = pd.read_csv("../data/raw_data.csv")
df_clean = raw.copy()

print("Raw shape:", raw.shape)
print("Raw exact duplicates:", raw.duplicated().sum())

Raw shape: (8020, 8)
Raw exact duplicates: 20


### Data Cleaning Decision Log

This log documents the data quality issues and the actions taken during cleaning. It explains the implemented decisions and the reasons for each treatment while preserving the raw files unchanged.

### Cleaning decisions

| Identified issue | Action | Rationale |
|---|---|---|
| 20 fully duplicated records | Remove duplicate records | Avoid counting the same reading more than once |
| Recording timestamps stored as text | Convert to datetime | Enable temporal inspection and analysis |
| Inconsistent spelling of sites and operating statuses | Trim whitespace and standardize case | Prevent duplicate categories caused by formatting differences |
| Thousands separators in some RPM readings | Remove commas and convert to numeric values | Enable correct numerical calculations |
| 40 missing measurement cells | Preserve missing values without imputation | Avoid creating unverified measurements |
| Outlying measurements | Retain values without deletion | No evidence establishes that these values are errors |
| Gaps in recording times | Do not generate replacement readings | It is uncertain whether the estimated readings were actually expected |

### Columns involved in cleaning

- `reading_at` — reading date and time
- `device_id` — device identifier used to order records
- `site` — site label whose formatting is standardized
- `status` — operating status whose categories are standardized
- `rpm` — device rotational speed whose formatting is standardized

### Code elements

- `drop_duplicates()` — remove fully identical records
- `pd.to_datetime()` — convert recording times to datetime
- `str.strip()` — remove leading and trailing whitespace
- `str.upper()` — standardize English letters to uppercase
- `str.replace()` — remove thousands separators from RPM values
- `pd.to_numeric()` — convert RPM readings to numeric values
- `sort_values()` — order records by the selected columns

### Ordering and saving the results

After cleaning, records are sorted by recording date and time, then by device identifier. The cleaned data is saved in a file separate from the raw data files.

### Data files

| File | Purpose |
|---|---|
| `original_data.csv` | Original data source |
| `raw_data.csv` | Raw working copy before cleaning |
| `cleaned_data.csv` | Resulting cleaned data |

### Methodological note

Only fully duplicated records are removed; readings are not deleted merely because their values are high. Missing values and outliers are retained without imputation or modification. No replacement readings are generated for the detected time gaps. Treatments are limited to documented procedures, and the raw files remain unchanged. Resolving formatting issues and duplicates does not establish that all measurements are operationally correct.


## Perform Data Cleaning

Cleaning is carried out on an independent copy of the raw data. This addresses the identified quality issues and prepares the dataset for subsequent analysis and validation.

### Cleaning steps

First, remove fully identical records while retaining the first occurrence of each record. Convert recording times to a suitable temporal data type. Standardize site names and operating-status labels by trimming whitespace and converting English letters to uppercase. Normalize RPM readings by removing thousands separators and converting values to numbers. Finally, sort records by time and device identifier, and reset the row index.

### Columns used

- `reading_at` — reading date and time
- `device_id` — device identifier used to order records
- `site` — site label whose formatting is standardized
- `status` — operating status whose labels are standardized
- `rpm` — device rotational speed converted to numeric form

### Variables

- `df_clean` — data copy used for cleaning
- `before_dedup` — record count before duplicate removal
- `removed_duplicates` — number of duplicate records removed

### Code elements

- `len()` — count records before and after removal
- `drop_duplicates()` — remove fully identical records
- `keep="first"` — retain the first occurrence of each record
- `copy()` — create an independent data copy
- `pd.to_datetime()` — convert recording times to datetime
- `errors="coerce"` — represent values that cannot be converted as missing
- `astype("string")` — convert values to strings during normalization
- `str.strip()` — remove leading and trailing whitespace
- `str.upper()` — standardize English letters to uppercase
- `str.replace()` — remove thousands separators from RPM readings
- `regex=False` — treat the comma as literal text
- `pd.to_numeric()` — convert RPM readings to numbers
- `sort_values()` — order records by recording time and device identifier
- `reset_index(drop=True)` — reset the row index without retaining the old index
- `shape` — display the dimensions after cleaning
- `print()` — display execution results

### Results

| Metric | Result |
|---|---:|
| Records before cleaning | 8,020 |
| Duplicate records removed | 20 |
| Records after cleaning | 8,000 |
| Columns after cleaning | 8 |

### Interpretation

The output shows that twenty fully duplicated records were removed, reducing the dataset to eight thousand rows. The number of columns is unchanged. These results confirm successful duplicate removal; the remaining treatments require the quality checks that follow.

### Methodological note

Treatments are applied to a separate copy without modifying the raw files. Missing values are not imputed, outliers are not deleted, and replacement readings are not generated for the previously identified time gaps. Numerical and datetime conversion allow unconvertible values to become missing, so conversion results must be checked for additional missing values. Successful cleaning does not establish that every measurement is operationally correct.


In [2]:
# 1 Remove exact duplicate rows
before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(keep="first").copy()
removed_duplicates = before_dedup - len(df_clean)

# 2 Convert timestamp
df_clean["reading_at"] = pd.to_datetime(
    df_clean["reading_at"], errors="coerce"
)

# 3 Normalize categories
df_clean["site"] = df_clean["site"].astype("string").str.strip().str.upper()
df_clean["status"] = df_clean["status"].astype("string").str.strip().str.upper()

# 4 Convert RPM to numeric
df_clean["rpm"] = pd.to_numeric(
    df_clean["rpm"].astype("string").str.replace(",", "", regex=False),
    errors="coerce"
)

# 5 Sort records
df_clean = df_clean.sort_values(
    ["reading_at", "device_id"]
).reset_index(drop=True)

print("Removed exact duplicates:", removed_duplicates)
print("Cleaned shape:", df_clean.shape)

Removed exact duplicates: 20
Cleaned shape: (8000, 8)


### Assess Data Quality After Cleaning

This step checks data quality after the documented cleaning operations. It verifies the treatments and compares the dataset's condition before and after cleaning.

### Inspection steps

Count records before and after cleaning to assess the effect of duplicate removal. Check that no fully identical records remain. Verify datetime validity and the conversion of RPM readings to numeric values. Inspect the unique site and operating-status categories after normalization. Finally, count missing values in the four measurement columns to confirm that they were retained without imputation.

### Columns used

- `reading_at` — recording date and time
- `site` — recorded site
- `status` — operating status
- `temp_c` — temperature
- `humidity_pct` — humidity percentage
- `vibration_mm_s` — vibration measurement
- `rpm` — device rotational speed

### Variables

- `raw` — raw data before cleaning
- `df_clean` — resulting cleaned data
- `value_cols` — four measurement columns checked for missing values
- `rpm_normalized_check` — result of converting raw RPM values after normalization
- `non_numeric_rpm_after` — originally non-missing values that still fail conversion after normalization
- `missing_rpm_after` — missing RPM values after cleaning
- `validation` — summary of data quality validation metrics

### Code elements

- `pd.to_numeric()` — test RPM conversion to numbers
- `astype("string")` — temporarily convert values to text
- `str.replace()` — remove thousands separators from RPM readings
- `errors="coerce"` — represent conversion failures as missing values
- `notna()` — identify non-missing values
- `isna()` — identify missing values
- `duplicated()` — check for fully identical records
- `sum()` — count cases satisfying the check conditions
- `len()` — count records
- `dropna()` — exclude missing values from category checks
- `unique()` — extract unique categories
- `sorted()` — sort categories
- `pd.Series()` — present validation metrics in an organized form

### Results

| Metric | Result |
|---|---:|
| Records before cleaning | 8,020 |
| Records after cleaning | 8,000 |
| Duplicate records after cleaning | 0 |
| Invalid dates after cleaning | 0 |
| RPM values that failed conversion after normalization | 0 |
| Missing RPM values | 10 |
| Total missing measurement cells | 40 |

### Categories after normalization

| Type | Resulting values |
|---|---|
| Sites | `NOR`, `RIV`, `SLT` |
| Operating statuses | `FAULT`, `OK`, `WARN` |

### Interpretation

The record count decreased after fully identical rows were removed. No duplicate records or invalid dates remain in the cleaned data. The RPM check found no additional conversion failures after thousands separators were removed. Site and operating-status categories are now standardized. Missing values are preserved without imputation or replacement measurements.

### Main findings

- Fully duplicated records were successfully removed
- No invalid dates appeared after cleaning
- Available RPM values were successfully converted after normalization
- Site and operating-status categories were standardized
- Missing values were retained without imputation

### Methodological note

The numeric-conversion check uses raw values after temporary normalization; the other validation metrics use the cleaned copy. Passing these checks does not prove operational correctness of all measurements. It also does not, on its own, establish that no unintended change occurred in other values. Missing values were not imputed, and no replacement readings were generated during cleaning.


In [3]:

# Validate cleaned dataset

value_cols = ["temp_c", "humidity_pct", "vibration_mm_s", "rpm"]

# Check RPM conversion using original raw values
rpm_normalized_check = pd.to_numeric(
    raw["rpm"].astype("string").str.replace(
        ",", "", regex=False
    ),
    errors="coerce"
)

non_numeric_rpm_after = int(
    (
        raw["rpm"].notna()
        & rpm_normalized_check.isna()
    ).sum()
)

missing_rpm_after = int(df_clean["rpm"].isna().sum())


# Main validation checks
validation = {
    "rows_before": len(raw),
    "rows_after": len(df_clean),

    "exact_duplicates_after": int(
        df_clean.duplicated().sum()
    ),

    "invalid_datetime_after": int(
        df_clean["reading_at"].isna().sum()
    ),

    "non_numeric_rpm_after": non_numeric_rpm_after,
    "missing_rpm_after": missing_rpm_after,

    "site_categories_after": sorted(
        df_clean["site"].dropna().unique().tolist()
    ),

    "status_categories_after": sorted(
        df_clean["status"].dropna().unique().tolist()
    ),

    "missing_measurement_cells_after": int(
        df_clean[value_cols].isna().sum().sum()
    )
}

pd.Series(validation, name="value")


rows_before                                     8020
rows_after                                      8000
exact_duplicates_after                             0
invalid_datetime_after                             0
non_numeric_rpm_after                              0
missing_rpm_after                                 10
site_categories_after                [NOR, RIV, SLT]
status_categories_after            [FAULT, OK, WARN]
missing_measurement_cells_after                   40
Name: value, dtype: object

### Validate Device Consistency, Time Ordering, and Measurement Types

Additional checks examine consistency after cleaning: device-to-site associations, chronological ordering, and whether measurement-column types are suitable for numerical calculations.

### Inspection steps

Group records by device identifier and count the unique sites associated with each device. Identify devices whose readings appear at more than one site. Check whether recording times are in ascending order. Finally, display the data types of the four numeric measurement columns.

### Columns used

- `device_id` — device identifier
- `site` — site where the device is recorded
- `reading_at` — recording date and time
- `temp_c` — temperature
- `humidity_pct` — humidity percentage
- `vibration_mm_s` — vibration measurement
- `rpm` — device rotational speed

### Variables

- `df_clean` — cleaned data copy
- `device_site_counts` — number of unique sites associated with each device
- `numeric_cols` — list of numeric measurement columns

### Code elements

- `groupby()` — group records by device identifier
- `nunique()` — count unique sites per device
- `sum()` — count devices associated with more than one site
- `is_monotonic_increasing` — verify ascending chronological order
- `dtypes` — display selected column data types
- `print()` — display check results

### Results

| Metric | Result |
|---|---:|
| Devices associated with more than one site | 0 |
| Records in chronological order | True |

### Measurement types after cleaning

| Column | Data type |
|---|---|
| `temp_c` | `float64` |
| `humidity_pct` | `float64` |
| `vibration_mm_s` | `float64` |
| `rpm` | `Float64` |

### Interpretation

No device is associated with more than one site in the cleaned dataset. Recording times are in ascending order. All four measurement columns have numeric types suitable for calculations.

### Methodological note

The site check counts non-missing unique values per device. Correct chronological ordering does not necessarily mean that all expected readings are present. The RPM column uses a numeric type that supports missing values. These checks alone do not establish operational correctness of the measurements. No data is modified during these additional validation checks.


In [4]:
# Validate one site per device
device_site_counts = df_clean.groupby("device_id")["site"].nunique()
print("Devices linked to more than one site:", int((device_site_counts > 1).sum()))

# Validate chronological ordering
print("Chronologically sorted:", df_clean["reading_at"].is_monotonic_increasing)

# Validate numeric measurement columns
numeric_cols = ["temp_c", "humidity_pct", "vibration_mm_s", "rpm"]
print(df_clean[numeric_cols].dtypes)

Devices linked to more than one site: 0
Chronologically sorted: True
temp_c            float64
humidity_pct      float64
vibration_mm_s    float64
rpm               Float64
dtype: object


### Compare Missing Values Before and After Cleaning

This step compares missing-value counts before and after cleaning to verify that missing values were preserved without imputation or unverified replacement measurements.

### Inspection steps

Select the four numeric measurement columns. Count missing cells in each raw-data column, repeat the check on the cleaned copy, and calculate the difference to identify changes in the counts.

### Columns used

- `temp_c` — temperature
- `humidity_pct` — humidity percentage
- `vibration_mm_s` — vibration measurement
- `rpm` — device rotational speed

### Variables

- `raw` — raw data before cleaning
- `df_clean` — data after cleaning
- `value_cols` — list of the four measurement columns
- `missing_comparison` — missing-value comparison table

### Code elements

- `pd.DataFrame()` — create a table of comparison results
- `isna()` — identify missing cells
- `sum()` — count missing values per column

### Results

| Measurement | Before cleaning | After cleaning | Change |
|---|---:|---:|---:|
| Temperature | 10 | 10 | 0 |
| Humidity | 10 | 10 | 0 |
| Vibration | 10 | 10 | 0 |
| Rotational speed | 10 | 10 | 0 |
| **Total** | **40** | **40** | **0** |

### Interpretation

Missing-value counts are unchanged in all four measurement columns. No increase or decrease occurred after cleaning, consistent with the project's decision to preserve missing values without imputation.

### Methodological note

This check compares the number of missing cells, not their locations within records. Equal counts alone do not prove that the missing-value locations are unchanged. Removing fully duplicated records did not alter the total missing-cell count in this dataset. No missing values were imputed and no replacement measurements were created during cleaning.


In [5]:
missing_comparison = pd.DataFrame({
    "raw": raw[value_cols].isna().sum(),
    "cleaned": df_clean[value_cols].isna().sum()
})
missing_comparison["change"] = missing_comparison["cleaned"] - missing_comparison["raw"]
missing_comparison

,raw,cleaned,change
temp_c,10,10,0
humidity_pct,10,10,0
vibration_mm_s,10,10,0
rpm,10,10,0


## Validate Data Integrity Before Saving

A set of validation checks is performed after cleaning and before saving the final dataset. These checks prevent saving if any specified quality condition is not met.

### Inspection steps

Verify that the cleaned record count matches the expected count after duplicate removal and that no fully identical records remain. Check recording-time validity and ascending chronological order. Confirm that RPM values converted successfully after normalization. Review missing-value counts to ensure they did not change during cleaning. Verify that each device is associated with at most one site in the available data and that all four measurement columns have numeric types. Finally, check category formatting and the expected site and operating-status values.

### Physical-range conditions

Additional checks verify that selected measurements lie within possible ranges.

| Measurement | Validation condition |
|---|---|
| Humidity | Between 0 and 100 |
| Vibration | Greater than or equal to 0 |
| Rotational speed | Greater than or equal to 0 |

Missing values are excluded from the physical-range checks only. Negative temperatures are not prohibited because they can be possible on the Celsius scale.

### Columns used

- `reading_at` — recording date and time
- `device_id` — device identifier
- `site` — device's recorded site
- `status` — operating status
- `temp_c` — temperature
- `humidity_pct` — humidity percentage
- `vibration_mm_s` — vibration measurement
- `rpm` — device rotational speed

### Variables

- `raw` — raw data before cleaning
- `df_clean` — resulting cleaned data
- `removed_duplicates` — number of duplicate records removed
- `non_numeric_rpm_after` — values that failed conversion after normalization
- `missing_comparison` — missing-value comparison before and after cleaning
- `device_site_counts` — unique site count for each device
- `value_cols` — list of numeric measurement columns

### Code elements

- `assert` — stop execution if a check fails
- `len()` — count records
- `duplicated()` — check for fully identical records
- `any()` — test whether a matching case exists
- `notna()` — identify non-missing values
- `all()` — verify a condition for every checked value
- `is_monotonic_increasing` — verify ascending time order
- `eq(0)` — check that missing-value counts did not change
- `le(1)` — ensure that no device is associated with more than one site
- `pd.api.types.is_numeric_dtype()` — check numeric data types
- `dropna()` — exclude missing values from the specified check
- `str.strip()` — remove excess whitespace
- `str.upper()` — standardize English letter case
- `set()` — verify the expected category sets
- `between()` — test whether values fall within a range
- `ge(0)` — check for non-negative values
- `print()` — display the validation success message

### Validation result

If all conditions pass, the following message is displayed:

`Pre-save validation: PASSED`

If any condition fails, execution stops and an error message explains the failure.

### Methodological note

Validation prevents saving data that does not satisfy the specified conditions. Conversion and category checks depend on the cleaning results from earlier steps. Equal missing-value counts alone do not prove identical missing-value locations. Physical-range checks exclude missing values without imputing them. These are basic consistency conditions, not a comprehensive engineering validation of measurement accuracy. Passing all conditions does not establish that the devices are free of mechanical faults. No records are removed and no values are changed by the pre-save validation checks.


In [6]:

# Pre-Save Data Validation

# 1) Validate row count and duplicates
assert len(df_clean) == len(raw) - removed_duplicates, (
    "Unexpected row count after cleaning"
)

assert not df_clean.duplicated().any(), (
    "Exact duplicate rows still exist"
)

# 2) Validate timestamps and chronological order
assert df_clean["reading_at"].notna().all(), (
    "Invalid timestamps found"
)

assert df_clean["reading_at"].is_monotonic_increasing, (
    "Records are not chronologically sorted"
)

# 3) Validate RPM conversion
assert non_numeric_rpm_after == 0, (
    "Some RPM values could not be converted"
)

# 4) Ensure missing values were not changed
assert missing_comparison["change"].eq(0).all(), (
    "Missing measurement counts have changed"
)

# 5) Ensure each device belongs to one site
assert device_site_counts.le(1).all(), (
    "Some devices belong to multiple sites"
)

# 6) Validate numeric measurement columns
for col in value_cols:
    assert pd.api.types.is_numeric_dtype(df_clean[col]), (
        f"{col} must be numeric"
    )

# 7) Validate category formatting
for col in ["site", "status"]:
    values = df_clean[col].dropna()

    assert values.eq(
        values.str.strip().str.upper()
    ).all(), f"{col} has inconsistent formatting"

# 8) Validate expected categories
assert set(df_clean["site"].dropna()) == {
    "NOR", "RIV", "SLT"
}, "Unexpected site category"

assert set(df_clean["status"].dropna()) == {
    "OK", "WARN", "FAULT"
}, "Unexpected status category"


# 9) Validate physical measurement ranges

assert df_clean["humidity_pct"].dropna().between(0, 100).all(), (
    "Humidity values must be between 0 and 100"
)

for col in ["vibration_mm_s", "rpm"]:
    assert df_clean[col].dropna().ge(0).all(), (
        f"{col} contains negative values"
    )

print("Pre-save validation: PASSED")


Pre-save validation: PASSED


## Validate Missing-Value Locations Before Saving

An additional check verifies that missing-value locations did not change during cleaning. Missing values should remain in the same records and columns before and after cleaning.

### Inspection steps

Create an independent raw-data copy after excluding fully identical records. Convert recording times to a type suitable for temporal comparison. Define a record key consisting of reading time and device identifier, and use it to align raw and cleaned records. Identify missing cells in the four measurement columns of both copies. Sort by the key so that results are ordered consistently, then compare missing-value locations.

### Columns used

- `reading_at` — reading date and time
- `device_id` — device identifier
- `temp_c` — temperature
- `humidity_pct` — humidity percentage
- `vibration_mm_s` — vibration measurement
- `rpm` — device rotational speed

### Variables

- `raw` — raw data before cleaning
- `df_clean` — resulting cleaned data
- `raw_unique` — raw-data copy after duplicate removal
- `record_key` — columns used to match records
- `value_cols` — list of the four measurement columns
- `raw_missing_positions` — missing-value locations in deduplicated raw data
- `clean_missing_positions` — missing-value locations in cleaned data

### Code elements

- `drop_duplicates()` — exclude fully identical records
- `keep="first"` — retain the first occurrence of each record
- `copy()` — create an independent data copy
- `pd.to_datetime()` — convert recording times to datetime
- `errors="raise"` — raise an error if datetime conversion fails
- `set_index()` — use recording time and device identifier as a comparison key
- `sort_index()` — sort records by that key
- `isna()` — identify missing-cell locations
- `pd.testing.assert_frame_equal()` — verify equal check results between copies
- `print()` — display a success message when the comparison passes

### Validation result

When missing-value locations match between deduplicated raw data and cleaned data, the following message is displayed:

`Missing-value positions: PASSED`

A mismatch raises an error indicating that the results differ.

### Why this check matters

It confirms that cleaning did not move missing cells and provides a more precise check than comparing only the total missing-value counts.

### Methodological note

The comparison is performed after removing fully identical records from the raw copy. It aligns records using recording time and device identifier, assuming this key identifies records appropriately for comparison. Only missing-value locations are compared, not non-missing numeric values. Passing this check alone therefore does not establish that every measurement value remained unchanged. Missing values are not imputed and the original data is not modified during this check.


In [7]:

# Verify missing-value positions before saving

raw_unique = raw.drop_duplicates(keep="first").copy()

raw_unique["reading_at"] = pd.to_datetime(
    raw_unique["reading_at"],
    errors="raise"
)

record_key = ["reading_at", "device_id"]

raw_missing_positions = (
    raw_unique
    .set_index(record_key)[value_cols]
    .sort_index()
    .isna()
)

clean_missing_positions = (
    df_clean
    .set_index(record_key)[value_cols]
    .sort_index()
    .isna()
)

pd.testing.assert_frame_equal(
    raw_missing_positions,
    clean_missing_positions
)

print("Missing-value positions: PASSED")


Missing-value positions: PASSED


## Validate the Cleaned Data Against the Original Source

A final comparison between the cleaned data and the original project source checks record alignment, category consistency, and preservation of available measurement values before the cleaned data is saved.

### Inspection steps

Load the original source and parse recording times as datetime. Define a record-matching key using recording time and device identifier, and verify that this key is unique in both the original and cleaned datasets. Merge both copies to check that all records match, without additional or unmatched records. Compare site and operating-status labels after standardizing their formatting in both copies. Temporarily normalize the numerical measurements and compare them, restricting this comparison to readings that are non-missing in the cleaned data. Finally, display the result for each of the four measurement columns.

### Files used

| File | Purpose |
|---|---|
| `original_data.csv` | Original source used for comparison |
| `cleaned_data.csv` | Name of the cleaned-data file to be saved |

### Columns used

- `reading_at` — recording date and time
- `device_id` — device identifier
- `site` — recorded site
- `status` — operating status
- `temp_c` — temperature
- `humidity_pct` — humidity percentage
- `vibration_mm_s` — vibration measurement
- `rpm` — device rotational speed

### Variables

- `original` — loaded original-source data
- `df_clean` — cleaned data copy
- `key` — columns used to match records
- `comparison` — comparison table joining the original and cleaned data
- `original_values` — original-source values extracted for comparison
- `cleaned_values` — cleaned-data values extracted for comparison
- `present` — mask identifying non-missing cleaned measurements
- `value_cols` — four numeric measurement columns

### Code elements

- `pd.read_csv()` — load the original source
- `parse_dates` — convert the recording-time column during loading
- `duplicated()` — check record-key uniqueness
- `merge()` — match original-source records to cleaned records
- `how="outer"` — include records from both copies in the comparison
- `indicator=True` — identify each merged record's source
- `validate="one_to_one"` — ensure matching keys are not duplicated during merging
- `astype("string")` — convert values to strings for comparison
- `str.strip()` — trim category whitespace
- `str.upper()` — standardize English letter case
- `str.replace()` — remove thousands separators from measurements
- `pd.to_numeric()` — convert measurements to numeric values
- `notna()` — identify non-missing readings
- `equals()` — compare normalized category values
- `np.allclose()` — compare available numeric values while allowing small numerical differences
- `assert` — stop execution when a validation condition fails
- `print()` — display comparison results

### Results

Record matching and site/status category comparisons passed. Available numeric values also matched the corresponding original-source values.

| Measurement column | Available-value comparison | Missing values after cleaning |
|---|---|---:|
| `temp_c` | Match within comparison tolerance | 10 |
| `humidity_pct` | Match within comparison tolerance | 10 |
| `vibration_mm_s` | Match within comparison tolerance | 10 |
| `rpm` | Match within comparison tolerance | 10 |
| **Total** | | **40** |

### Final validation result

All implemented comparison conditions passed, producing:

`Original vs cleaned: PASSED`

### Interpretation

Record keys match between the original and cleaned datasets. Site and operating-status values match after formatting normalization. No differences beyond the specified numerical tolerance were found among measurements available in the cleaned data. Missing values remain excluded from the numeric-value comparison with the original source.

### Methodological note

The original source is used only for validation and comparison, not to fill missing values in the working copy. Numeric comparisons use non-missing cleaned measurements, so passing the check does not establish that all cells match across the two copies. Small numerical differences are permitted by the comparison settings. The stability of missing-value locations between deduplicated raw data and cleaned data was checked earlier. These checks alone do not establish operational correctness of the measurements. No missing values are imputed and the original source is not modified during this comparison.


In [ ]:

# Compare cleaned data with the original source
original = pd.read_csv(
    "../data/original_data.csv",
    parse_dates=["reading_at"]
)

key = ["reading_at", "device_id"]

# Check unique record identifiers
assert not original.duplicated(key).any()
assert not df_clean.duplicated(key).any()

# Match records by device and timestamp
comparison = original.merge(
    df_clean,
    on=key,
    suffixes=("_original", "_cleaned"),
    how="outer",
    indicator=True,
    validate="one_to_one"
)

assert comparison["_merge"].eq("both").all()

# Compare categorical columns
for col in ["site", "status"]:
    original_values = (
        comparison[f"{col}_original"]
        .astype("string").str.strip().str.upper()
    )

    cleaned_values = (
        comparison[f"{col}_cleaned"]
        .astype("string").str.strip().str.upper()
    )

    assert original_values.equals(cleaned_values)

# Compare existing measurement values
for col in value_cols:
    original_values = pd.to_numeric(
        comparison[f"{col}_original"]
        .astype("string")
        .str.replace(",", "", regex=False),
        errors="coerce"
    )

    cleaned_values = pd.to_numeric(
        comparison[f"{col}_cleaned"]
        .astype("string")
        .str.replace(",", "", regex=False),
        errors="coerce"
    )

    present = cleaned_values.notna()

    assert original_values.notna().all()

    assert np.allclose(
        original_values[present].to_numpy(dtype=float),
        cleaned_values[present].to_numpy(dtype=float),
        rtol=1e-12,
        atol=1e-12
    )

    print(
        f"{col}: values match; missing = {int((~present).sum())}"
    )

print("Original vs cleaned: PASSED")


temp_c: values match; missing = 10
humidity_pct: values match; missing = 10
vibration_mm_s: values match; missing = 10
rpm: values match; missing = 10
Original vs cleaned: PASSED


## Save the Cleaned Dataset

After the preceding validation checks pass, the cleaned data is saved to a separate file. This preserves the original source and raw working copy unchanged.

### Saving steps

Define the destination path in the project's data directory. Export the cleaned copy as a comma-separated values file without adding the row index as a data column. Finally, display the saved path and the resulting dataset dimensions.

### Data files

| File | Rows | Purpose |
|---|---:|---|
| `original_data.csv` | 8,000 | Original source |
| `raw_data.csv` | 8,020 | Raw working copy |
| `cleaned_data.csv` | 8,000 | Cleaned data |

### Variables

- `df_clean` — data after cleaning and validation
- `output_path` — destination path for the cleaned-data file

### Code elements

- `Path()` — define the save path
- `to_csv()` — export the data to a file
- `index=False` — omit the row index from the file
- `shape` — display the cleaned dimensions
- `print()` — display the saved path and results

### Results

| Metric | Result |
|---|---:|
| Records saved | 8,000 |
| Columns saved | 8 |
| Duplicate records removed earlier | 20 |

### Interpretation

The output confirms successful export to the specified file. The number of columns remains unchanged after duplicate removal. The cleaned data is now stored separately from the original and raw copies.

### Methodological note

The data is saved after validation, without imputing missing values or removing outliers. Neither the original source nor the raw working copy is modified during saving. This step confirms that export occurred; the saved file's contents are checked in the next step after reloading.


In [9]:
output_path = Path("../data/cleaned_data.csv")
df_clean.to_csv(output_path, index=False)
print("Saved:", output_path)
print("Final shape:", df_clean.shape)

Saved: ..\data\cleaned_data.csv
Final shape: (8000, 8)


### Additional Data-Type Validation

After the cleaned file is reloaded, an additional check validates its column types. It verifies that datetime, numeric, and categorical columns have the required types in the loaded data.

### Columns checked

| Column | Required type |
|---|---|
| `reading_at` | `Datetime` |
| `temp_c` | `Numeric` |
| `humidity_pct` | `Numeric` |
| `vibration_mm_s` | `Numeric` |
| `rpm` | `Numeric` |
| `site` | `String` |
| `status` | `String` |

### Code elements

| Element | Purpose |
|---|---|
| `is_datetime64_any_dtype()` | Validate the datetime type |
| `is_numeric_dtype()` | Validate numeric types |
| `is_string_dtype()` | Validate string types |
| `assert` | Stop execution if a condition fails |
| `isna().sum().sum()` | Count all missing values |

### Additional validation results

| Check | Required result |
|---|---|
| Datetime type validation | `PASSED` |
| Numeric column validation | `PASSED` |
| String column validation | `PASSED` |
| Missing-value count validation | `PASSED` |

### Methodological note

These checks reflect the types after reloading with the specified read settings. Dates are converted and certain column types are assigned while reading the saved file. Passing therefore verifies the types of the reloaded data under those settings. The `CSV` format is textual and does not independently preserve programming-language data types; column types are determined on reload. Passing type checks does not establish operational correctness of all measurements. Original data and values are not modified during this check.


In [10]:

# Verify Saved Cleaned Dataset
saved_path = Path("../data/cleaned_data.csv")

print("File exists:", saved_path.exists())

saved_df = pd.read_csv(
    
    saved_path,
    parse_dates=["reading_at"],
    dtype={
        "site": "string",
        "status": "string",
        "rpm": "Float64"
    }
)

print("Saved dataset shape:", saved_df.shape)
print("Original cleaned shape:", df_clean.shape)

print("Shape matches:", saved_df.shape == df_clean.shape)

print(
    "Columns match:",
    saved_df.columns.tolist() == df_clean.columns.tolist()
)

print(
    "Missing measurement cells:",
    saved_df[value_cols].isna().sum().sum()
)

# Verify actual data values
pd.testing.assert_frame_equal(

    saved_df,
    df_clean,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12
)

print("Data integrity check: PASSED")

# Verify data types after loading the saved CSV
assert pd.api.types.is_datetime64_any_dtype(
    saved_df["reading_at"]
)

for col in value_cols:
    assert pd.api.types.is_numeric_dtype(
        saved_df[col]
    ), f"Invalid numeric type: {col}"

for col in ["site", "status"]:
    assert pd.api.types.is_string_dtype(
        saved_df[col]
    ), f"Invalid category type: {col}"

assert (
    saved_df[value_cols].isna().sum().sum()
    == df_clean[value_cols].isna().sum().sum()
)

print("Datetime validation: PASSED")
print("Numeric columns validation: PASSED")
print("Category columns validation: PASSED")
print("Missing values validation: PASSED")



File exists: True
Saved dataset shape: (8000, 8)
Original cleaned shape: (8000, 8)
Shape matches: True
Columns match: True
Missing measurement cells: 40
Data integrity check: PASSED
Datetime validation: PASSED
Numeric columns validation: PASSED
Category columns validation: PASSED
Missing values validation: PASSED


### Cross-Check Data Equivalence

An additional comparison checks data reconstructed from the raw copy against the saved cleaned dataset. It verifies that removing duplicates and normalizing formats did not cause unexplained differences in records or measurements.

### Inspection steps

Create a temporary raw-data copy without fully duplicated records. Convert the date column to datetime and standardize site and operating-status labels. Convert RPM to numeric values after removing commas from affected values. Verify that the reading-time/device-identifier key is unique in each copy. Sort both copies by the same key and reset their indexes for a consistent comparison. Finally, compare every column and value while allowing very small numerical differences and ignoring data-type differences.

### Variables

| Variable | Description |
|---|---|
| `raw` | Raw data used in the cleaning notebook |
| `analysis_comparison` | Temporary copy reconstructed from raw data for comparison |
| `saved_df` | Cleaned data reloaded from the saved file |
| `keys` | Columns used to identify and sort records |
| `left` | Reconstructed data after sorting |
| `right` | Saved cleaned data after sorting |

### Code elements

| Element | Purpose |
|---|---|
| `drop_duplicates()` | Exclude fully duplicated records |
| `pd.to_datetime()` | Convert date values to datetime |
| `str.strip()` | Remove excess whitespace |
| `str.upper()` | Standardize English letter case |
| `pd.to_numeric()` | Convert RPM values to numeric form |
| `duplicated()` | Check for duplicate keys |
| `sort_values()` | Sort records before comparison |
| `reset_index()` | Reset the row index |
| `assert` | Stop execution if a condition is not met |
| `assert_frame_equal()` | Compare the structure and values of both copies |

### Results

| Metric | Result |
|---|---|
| Records compared | 8,000 |
| Column names and order match | `True` |
| Data equivalence under comparison conditions | `PASSED` |

### Interpretation

The reconstructed raw-data copy and the saved cleaned dataset passed the comparison. They contain equivalent records and values after the specified transformations and sorting. No differences beyond the numerical comparison tolerances were found.

### Methodological note

The comparison uses a temporary copy reconstructed from raw data with the stated transformations. It is not a direct comparison to the exploratory-analysis variable in the first notebook. The test permits data-type differences and very small numerical differences; passing therefore does not mean byte-for-byte identity of the files or identical storage and column types. Nor does it establish that all earlier analyses used the cleaned data directly. Neither the raw data nor the saved cleaned copy is modified during this comparison.


In [ ]:

# Cross-check analytical data against saved cleaned data
analysis_comparison = raw.drop_duplicates(keep="first").copy()

analysis_comparison["reading_at"] = pd.to_datetime(
    analysis_comparison["reading_at"], errors="raise"
)

for column in ["site", "status"]:
    analysis_comparison[column] = (
        analysis_comparison[column]
        .astype("string").str.strip().str.upper()
    )

analysis_comparison["rpm"] = pd.to_numeric(
    analysis_comparison["rpm"]
    .astype("string")
    .str.replace(",", "", regex=False),
    errors="coerce"
)

keys = ["reading_at", "device_id"]

assert not analysis_comparison.duplicated(keys).any()
assert not saved_df.duplicated(keys).any()

left = analysis_comparison.sort_values(keys).reset_index(drop=True)
right = saved_df.sort_values(keys).reset_index(drop=True)

pd.testing.assert_frame_equal(
    left,
    right,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12
)

print("Rows compared:", len(left))
print("Same columns:", left.columns.tolist() == right.columns.tolist())
print("Data equivalence: PASSED")


Rows compared: 8000
Same columns: True
Data equivalence: PASSED


## Stage Two Summary and Treatment Limitations

Factory sensor data was cleaned and validated before saving the final copy. Processing started from the raw working dataset while preserving the original source unchanged.

### Cleaning operations performed

- Removed 20 fully duplicated records while retaining the first occurrence
- Converted recording times to datetime
- Standardized site and operating-status labels
- Removed thousands separators and converted RPM to numeric values
- Sorted records by recording time and device identifier

### Decisions and limitations

- Missing values — retain every missing cell without imputation
- Outliers — retain readings because they have not been established as errors
- Time gaps — do not create replacement readings from uncertain estimates
- Duplicates — remove only fully identical records
- Categories — normalize spelling without changing the original meaning of operating statuses

### Final results

| Metric | Result |
|---|---:|
| Records before cleaning | 8,020 |
| Duplicate records removed | 20 |
| Records after cleaning | 8,000 |
| Columns | 8 |
| Missing cells retained | 40 |
| Potential unrecorded readings estimated from time gaps | 468 |
| Remaining duplicate records | 0 |

### Final validation checks

The checks confirmed valid dates, consistent categories, numeric data types, chronological ordering, and no device associated with more than one site. Missing-value locations were matched before and after cleaning to verify that they did not change. The cleaned data passed the comparison with the original source under the specified conditions. Finally, the cleaned file was saved and reloaded to verify its contents.

### Methodological note

The estimate of potential unrecorded readings assumes one reading every two hours. It is therefore not a confirmed count of missing records. Retaining outliers does not establish their operational correctness. Treatments remain limited to documented procedures, with no missing-value imputation or replacement readings.

### Final conclusion

Cleaning and validation are complete, and the cleaned copy passed the quality checks defined in this project. Results were saved in a separate file while preserving the original and raw data files.

**Final file:** `cleaned_data.csv`

The data is ready for subsequent analyses within the quality limitations and assumptions documented in this project.
